<a href="https://colab.research.google.com/github/3aLaee/machine-learning-project/blob/main/LGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/drive/My Drive/ML/train")

OUT_PATH = DATA_DIR / "merged_input_output_v2.csv"

print("DATA_DIR:", DATA_DIR)
print("Will save merged file to:", OUT_PATH)

DATA_DIR: /content/drive/My Drive/ML/train
Will save merged file to: /content/drive/My Drive/ML/train/merged_input_output_v2.csv


## Throw state + direction normalization + per-week merge

In [20]:
def get_throw_state(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (game_id, play_id, nfl_id) get the last pre-pass frame.
    Rename columns so we have x_curr, y_curr, frame_id_curr, etc.
    """
    throw_state = (
        input_df
        .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
        .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
        .tail(1)
    )

    throw_state = throw_state.rename(
        columns={
            "frame_id": "frame_id_curr",
            "x": "x_curr",
            "y": "y_curr",
        }
    )
    return throw_state


def normalize_direction(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make all plays go to the right.
    Flip x/y coordinates and ball landing for plays originally going left.
    Rotate dir and o by 180 degrees for left plays.
    """
    df = df.copy()
    mask = df["play_direction"] == "left"

    # Flip current position at throw
    df.loc[mask, "x_curr"]      = 120.0 - df.loc[mask, "x_curr"]
    df.loc[mask, "y_curr"]      = 53.3  - df.loc[mask, "y_curr"]

    # Flip future positions
    df.loc[mask, "x_future"]    = 120.0 - df.loc[mask, "x_future"]
    df.loc[mask, "y_future"]    = 53.3  - df.loc[mask, "y_future"]

    # Flip ball landing
    df.loc[mask, "ball_land_x"] = 120.0 - df.loc[mask, "ball_land_x"]
    df.loc[mask, "ball_land_y"] = 53.3  - df.loc[mask, "ball_land_y"]

    # Rotate movement / orientation by 180°
    if "dir" in df.columns:
        df.loc[mask, "dir"] = (df.loc[mask, "dir"] + 180.0) % 360.0
    if "o" in df.columns:
        df.loc[mask, "o"]   = (df.loc[mask, "o"]   + 180.0) % 360.0

    df.loc[mask, "play_direction"] = "right"
    return df


def build_week_dataset(week: int) -> pd.DataFrame:

    print(f"Processing week {week:02d}...")

    input_path = DATA_DIR / f"input_2023_w{week:02d}.csv"
    output_path = DATA_DIR / f"output_2023_w{week:02d}.csv"

    input_df = pd.read_csv(input_path)
    output_df = pd.read_csv(output_path)

    # Get last pre-pass frame per player
    throw_state = get_throw_state(input_df)

    # Merge throw state onto all post-pass frames
    week_df = output_df.merge(
        throw_state,
        on=["game_id", "play_id", "nfl_id"],
        how="inner",
        validate="many_to_one",
    )

    # Rename output x,y + frame_id
    week_df = week_df.rename(
        columns={
            "frame_id": "frame_offset",
            "x": "x_future",
            "y": "y_future",
        }
    )

    # Normalise play direction to 'right'
    week_df = normalize_direction(week_df)

    # Add week column
    week_df["week"] = week

    # frame_offset vs num_frames_output
    if "num_frames_output" in week_df.columns:
        bad = (week_df["frame_offset"] > week_df["num_frames_output"]).sum()
        if bad > 0:
            print(f"WARNING: {bad} rows with frame_offset > num_frames_output in week {week}")

    return week_df

## merged dataset for all weeks and save

In [21]:

all_weeks = []
for w in range(1, 19):
    week_df = build_week_dataset(w)
    all_weeks.append(week_df)

df_merged = pd.concat(all_weeks, ignore_index=True)
print("\nMerged dataset shape (all weeks):", df_merged.shape)


if "player_to_predict" in df_merged.columns:
    df_merged = df_merged[df_merged["player_to_predict"] == True].copy()
    print("After filtering player_to_predict == True:", df_merged.shape)


display(df_merged.head())

df_merged.to_csv(OUT_PATH, index=False)
print(f"\n Saved merged dataset to:\n{OUT_PATH}")

Processing week 01...
Processing week 02...
Processing week 03...
Processing week 04...
Processing week 05...
Processing week 06...
Processing week 07...
Processing week 08...
Processing week 09...
Processing week 10...
Processing week 11...
Processing week 12...
Processing week 13...
Processing week 14...
Processing week 15...
Processing week 16...
Processing week 17...
Processing week 18...

Merged dataset shape (all weeks): (562936, 27)
After filtering player_to_predict == True: (562936, 27)


,game_id,play_id,nfl_id,frame_offset,x_future,y_future,player_to_predict,frame_id_curr,play_direction,absolute_yardline_number,...,x_curr,y_curr,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,week
0,2023090700,101,46137,1,56.22,17.28,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
1,2023090700,101,46137,2,56.63,16.88,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
2,2023090700,101,46137,3,57.06,16.46,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
3,2023090700,101,46137,4,57.48,16.02,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
4,2023090700,101,46137,5,57.91,15.56,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1



 Saved merged dataset to:
/content/drive/My Drive/ML/train/merged_input_output_v2.csv


In [22]:
!pip install lightgbm

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

merged_path = Path("/content/drive/My Drive/ML/train/merged_input_output_v2.csv")

df_merged = pd.read_csv(merged_path)
print("Shape:", df_merged.shape)
df_merged.head()

Shape: (562936, 27)


,game_id,play_id,nfl_id,frame_offset,x_future,y_future,player_to_predict,frame_id_curr,play_direction,absolute_yardline_number,...,x_curr,y_curr,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,week
0,2023090700,101,46137,1,56.22,17.28,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
1,2023090700,101,46137,2,56.63,16.88,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
2,2023090700,101,46137,3,57.06,16.46,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
3,2023090700,101,46137,4,57.48,16.02,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1
4,2023090700,101,46137,5,57.91,15.56,True,26,right,42,...,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22,1


## Keep only scored players

In [24]:
if "player_to_predict" in df_merged.columns:
    before = df_merged.shape[0]
    df_merged = df_merged[df_merged["player_to_predict"] == True].copy()
    print("Rows before filtering:", before)
    print("Rows after  filtering:", df_merged.shape[0])
else:
    print("No 'player_to_predict' column, skipping filter.")


Rows before filtering: 562936
Rows after  filtering: 562936


## Convert numeric columns

In [25]:
numeric_cols_to_convert = [
    "x_curr", "y_curr", "ball_land_x", "ball_land_y",
    "x_future", "y_future", "s", "a", "dir", "o",
    "absolute_yardline_number", "frame_offset"
]

for col in numeric_cols_to_convert:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")
    else:
        print(f"WARNING: column {col} not found")

df_merged[numeric_cols_to_convert].describe().T

,count,mean,std,min,25%,50%,75%,max
x_curr,562936.0,67.020923,22.991527,7.440000,48.5100,64.520000,84.440000,119.390000
y_curr,562936.0,26.491414,11.787700,0.690000,16.2500,26.300000,36.800000,52.620000
ball_land_x,562936.0,71.686996,24.807835,7.800000,51.5700,69.860001,91.050003,125.849998
ball_land_y,562936.0,26.505650,16.508331,-4.030002,11.2700,26.150000,41.850000,57.210000
x_future,562936.0,69.507312,23.435529,7.460000,50.6900,67.370000,87.610000,120.830000
y_future,562936.0,26.491081,13.445279,0.300000,14.8100,26.280000,38.270000,53.720000
s,562936.0,4.811613,2.052455,0.000000,3.3000,4.770000,6.350000,10.340000
a,562936.0,2.732318,1.465647,0.000000,1.5800,2.550000,3.730000,8.360000
dir,562936.0,113.981276,78.567121,0.000000,59.7175,101.420000,150.600000,360.000000
o,562936.0,207.945443,100.249916,0.000000,134.6600,233.130000,285.530000,359.970000


## Feature engineering

In [26]:
# Direction in radians
df_merged["dir_rad"] = np.deg2rad(df_merged["dir"])

# Current velocity components at throw
df_merged["vx_curr"] = df_merged["s"] * np.cos(df_merged["dir_rad"])
df_merged["vy_curr"] = df_merged["s"] * np.sin(df_merged["dir_rad"])

# Distance from current position to ball landing
df_merged["dist_to_land_curr"] = np.sqrt(
    (df_merged["x_curr"] - df_merged["ball_land_x"])**2 +
    (df_merged["y_curr"] - df_merged["ball_land_y"])**2
)

# Angle from current position to ball landing
df_merged["angle_to_landing"] = np.arctan2(
    df_merged["ball_land_y"] - df_merged["y_curr"],
    df_merged["ball_land_x"] - df_merged["x_curr"]
)

# Time after throw in seconds (10 fps)
df_merged["t"] = df_merged["frame_offset"] / 10.0


## Define features, targets & clean NaNs

In [27]:
feature_cols = [
    "x_curr", "y_curr",
    "vx_curr", "vy_curr",
    "s", "a", "dir", "o",
    "ball_land_x", "ball_land_y",
    "dist_to_land_curr", "angle_to_landing",
    "absolute_yardline_number",
    "frame_offset",
    "t"
]

target_cols = ["x_future", "y_future"]
week_col = "week"

cols_to_check = feature_cols + target_cols + [week_col]

before = df_merged.shape[0]
df_clean = df_merged.dropna(subset=cols_to_check).copy()
after = df_clean.shape[0]

print("Rows before cleaning:", before)
print("Rows after cleaning :", after)


Rows before cleaning: 562936
Rows after cleaning : 562936


## Train split by week

In [28]:
train_weeks = list(range(1, 15))
val_weeks   = list(range(15, 19))

train_df = df_clean[df_clean[week_col].isin(train_weeks)]
val_df   = df_clean[df_clean[week_col].isin(val_weeks)]

X_train = train_df[feature_cols]
y_train_x = train_df["x_future"]
y_train_y = train_df["y_future"]

X_val = val_df[feature_cols]
y_val_x = val_df["x_future"]
y_val_y = val_df["y_future"]

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)


X_train: (430955, 15)
X_val  : (131981, 15)


## Train LightGBM models with early stopping

In [29]:
params = dict(
    objective="regression",
    learning_rate=0.05,
    n_estimators=1000,
    max_depth=-1,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
)

lgbm_x = lgb.LGBMRegressor(**params)
lgbm_y = lgb.LGBMRegressor(**params)

print("\nTraining LGBM for x_future...")
lgbm_x.fit(
    X_train, y_train_x,
    eval_set=[(X_val, y_val_x)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

print("\nTraining LGBM for y_future...")
lgbm_y.fit(
    X_train, y_train_y,
    eval_set=[(X_val, y_val_y)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)



Training LGBM for x_future...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.106147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3328
[LightGBM] [Info] Number of data points in the train set: 430955, number of used features: 15
[LightGBM] [Info] Start training from score 69.372933

Training LGBM for y_future...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.065033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3328
[LightGBM] [Info] Number of data points in the train set: 430955, number of used features: 15
[LightGBM] [Info] Start training from score 26.424482


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
              n_jobs=-1, num_leaves=64, objective='regression', random_state=42,
              subsample=0.8)

## RMSE (x, y and 2D)

In [30]:
# Predictions on validation set
y_pred_x = lgbm_x.predict(X_val)
y_pred_y = lgbm_y.predict(X_val)

# 1D RMSEs
rmse_x = np.sqrt(mean_squared_error(y_val_x, y_pred_x))
rmse_y = np.sqrt(mean_squared_error(y_val_y, y_pred_y))

# 2D RMSE
err2 = (y_val_x - y_pred_x) ** 2 + (y_val_y - y_pred_y) ** 2
rmse_2d = np.sqrt(err2.mean())

print("\n MODEL PERFORMANCE (Validation Set)")
print(f"RMSE for x_future: {rmse_x:.4f}")
print(f"RMSE for y_future: {rmse_y:.4f}")
print(f" Combined 2D RMSE (spatial error): {rmse_2d:.4f} yards")



 MODEL PERFORMANCE (Validation Set)
RMSE for x_future: 1.1286
RMSE for y_future: 1.0196
 Combined 2D RMSE (spatial error): 1.5210 yards


## Constant-velocity baseline

In [31]:
# Constant-velocity prediction: x_cv = x_curr + vx_curr * t, etc.
x_cv = val_df["x_curr"] + val_df["vx_curr"] * val_df["t"]
y_cv = val_df["y_curr"] + val_df["vy_curr"] * val_df["t"]

err2_cv = (y_val_x - x_cv) ** 2 + (y_val_y - y_cv) ** 2
rmse_2d_cv = np.sqrt(err2_cv.mean())

print("CV  2D RMSE:", rmse_2d_cv)
print("LGBM 2D RMSE:", rmse_2d)
print("Improvement factor (CV / LGBM):", rmse_2d_cv / rmse_2d)


CV  2D RMSE: 7.965469734289072
LGBM 2D RMSE: 1.5209797255855888
Improvement factor (CV / LGBM): 5.237065031371345
